# Combine LLM Classifiers Output

- After running `llm-classifiers.py` per LLM, each output is saved in `data/classification_results/[dataset_folder_name]/seed[...]/in_domain/`
    1. `checkpoint_[model_name].csv`: llm label (0: non-prediction, 1: prediction) per sample/sentence
    2. `metrics_summary_[model_name].csv`: test accuracy, precision, recall, etc

- Tasks: 
    1. Combine the `checkpoint_[model_name].csv` by having all models into joint csv.
    2. Average and standard deviation on `metrics_summary_[model_name].csv` across all seeds.

In [1]:
import os
import re
import sys

import pandas as pd

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 40)
# pd.set_option('display.max_rows', None)

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
dataset_folder_path = os.path.join(base_data_path, 'classification_results', 'synthetic-fpb-chronicle2050-yt-news-timebank_2026-04-22')

In [4]:
dir_files = os.listdir(dataset_folder_path)
dir_files

['seed3', 'averaged', 'seed7', 'seed33']

In [26]:
def load_seed_data(dataset_folder_path, dir_files):
    """
    Walk through all seed folders and load checkpoint, metrics, and test set files.
    Parameters
    ----------
    dataset_folder_path : str
        Base path to the experiment folder containing seed subfolders.
    dir_files : list of str
        List of folder/file names inside dataset_folder_path.
    Returns
    -------
    seed_checkpoint_map : dict
        {seed_value: [df1, df2, ...]} one DataFrame per model checkpoint per seed.
    metrics_dfs_by_seed : dict
        {seed_value: [df1, df2, ...]} one DataFrame per model metrics per seed.
    x_test_dfs : list of pd.DataFrame
        All x_y_test_set DataFrames per seed.
    """
    metrics_dfs_by_seed = {}  # <--- CHANGED from flat list to dict grouped by seed
    x_test_dfs = []
    seed_checkpoint_map = {}

    for dir_file in dir_files:
        if "seed" not in dir_file:
            continue

        seed_path = os.path.join(dataset_folder_path, dir_file, 'in_domain')
        seed_path_files = os.listdir(seed_path)
        seed_value = int(re.search(r'\d+', dir_file).group())

        if seed_value not in seed_checkpoint_map:
            seed_checkpoint_map[seed_value] = []

        if seed_value not in metrics_dfs_by_seed:  # <--- ADDED
            metrics_dfs_by_seed[seed_value] = []   # <--- ADDED

        for seed_path_file in seed_path_files:

            if "checkpoint" in seed_path_file:
                print(f"Checking if 'checkpoint' is in \n\t{seed_path_file}")
                checkpoint_path = os.path.join(seed_path, seed_path_file)
                df = DataProcessing.load_from_file(path=checkpoint_path)
                df[df.columns.to_list()[-1]] = df['llm_label'].values
                df.drop(columns=['llm_label', 'raw_response', 'llm_name'], inplace=True)
                seed_checkpoint_map[seed_value].append(df)

            if "metrics" in seed_path_file and "ml" not in seed_path_file:
                print(f"Checking if 'metrics' is in \n\t{seed_path_file}")
                metrics_path = os.path.join(seed_path, seed_path_file)
                df = DataProcessing.load_from_file(path=metrics_path)
                metrics_dfs_by_seed[seed_value].append(df)  # <--- CHANGED from flat append to dict append

            if "x_y_test_set" in seed_path_file:
                x_test_path = os.path.join(seed_path, 'x_y_test_set.csv')
                x_test_df = DataProcessing.load_from_file(x_test_path)
                x_test_dfs.append(x_test_df)

    return seed_checkpoint_map, metrics_dfs_by_seed, x_test_dfs


def combine_checkpoints_per_seed(seed_checkpoint_map, dataset_folder_path):
    """
    Combine all model checkpoint DataFrames per seed into one wide DataFrame.
    Each combined DataFrame has: seed, original_index, text, <model_1>, <model_2>, ...

    Parameters
    ----------
    seed_checkpoint_map : dict
        {seed_value: [df1, df2, ...]} one DataFrame per model per seed.
    dataset_folder_path : str
        Base path used to construct the save path for each seed.

    Returns
    -------
    seeds_dfs : list of pd.DataFrame
        seeds_dfs[0] = seed3 combined, seeds_dfs[1] = seed7, seeds_dfs[2] = seed33.
    """
    seeds_dfs = []

    for seed_value in sorted(seed_checkpoint_map.keys()):
        seed_model_dfs = seed_checkpoint_map[seed_value]

        if not seed_model_dfs:
            print(f"⚠️  No checkpoint files found for seed {seed_value}. Skipping.")
            continue

        combined_df = seed_model_dfs[0][['seed', 'original_index', 'text']].copy()

        for df in seed_model_dfs:
            model_col = df.columns.to_list()[-1]
            combined_df[model_col] = df[model_col].values

        # Replace NaN with 0 and force int so labels are 0/1 not 0.0/1.0
        combined_df = combined_df.fillna(0)
        model_cols = [col for col in combined_df.columns if col not in ['seed', 'original_index', 'text']]
        combined_df[model_cols] = combined_df[model_cols].astype(int)

        print(f"\n✓ Seed {seed_value} combined shape: {combined_df.shape}")
        print(f"  Columns: {combined_df.columns.to_list()}")

        save_path = os.path.join(dataset_folder_path, f'seed{seed_value}', 'in_domain')
        DataProcessing.save_to_file(
            data=combined_df,
            path=save_path,
            prefix='llm_classifiers_in_domain',
            save_file_type='csv',
            include_version=False
        )
        print(f"✓ Saved: {os.path.join(save_path, 'llm_classifiers_in_domain.csv')}")

        seeds_dfs.append(combined_df)

    print(f"\nTotal seeds combined: {len(seeds_dfs)}")
    print(f"seeds_dfs[0] -> seed {sorted(seed_checkpoint_map.keys())[0]}")

    return seeds_dfs


def combine_and_save_metrics(metrics_dfs_by_seed, seed_checkpoint_map, dataset_folder_path):
    """
    Combine metrics from different LLM models within the same seed into one file.
    Mirrors the ML pipeline's metrics_summary_ml_models.csv structure where
    each seed has one file with one row per model.

    Parameters
    ----------
    metrics_dfs_by_seed : dict
        {seed_value: [df1, df2, ...]} one DataFrame per model per seed.
    seed_checkpoint_map : dict
        Used to get sorted seed values for saving to correct folders.
    dataset_folder_path : str
        Base path used to construct the save path for each seed.

    Returns
    -------
    dict
        {seed_value: combined_metrics_df} one combined DataFrame per seed.
    """
    if not metrics_dfs_by_seed:
        print("⚠️  No metrics files found. Skipping metrics summary.")
        return None

    seed_metrics_combined = {}

    for seed_value in sorted(seed_checkpoint_map.keys()):
        seed_metrics = metrics_dfs_by_seed.get(seed_value, [])

        if not seed_metrics:
            print(f"⚠️  No metrics found for seed {seed_value}. Skipping.")
            continue

        # Stack all model metric rows for this seed into one DataFrame
        # One row per LLM model — mirrors metrics_summary_ml_models.csv
        combined_metrics_df = DataProcessing.concat_dfs(seed_metrics)

        print(f"\n✓ Seed {seed_value} metrics shape: {combined_metrics_df.shape}")
        print(f"\nPreview:\n{combined_metrics_df}\n")

        # Save alongside ML metrics for easy comparison
        save_metrics_path = os.path.join(dataset_folder_path, f'seed{seed_value}', 'in_domain')
        DataProcessing.save_to_file(
            data=combined_metrics_df,
            path=save_metrics_path,
            prefix='metrics_summary_llms',
            save_file_type='csv',
            include_version=False
        )
        print(f"✓ Saved metrics_summary_llms.csv to: {save_metrics_path}")

        seed_metrics_combined[seed_value] = combined_metrics_df

    return seed_metrics_combined

In [27]:
seed_checkpoint_map, metrics_dfs_by_seed, x_test_dfs = load_seed_data(dataset_folder_path, dir_files)
seeds_dfs = combine_checkpoints_per_seed(seed_checkpoint_map, dataset_folder_path)
seed_metrics_combined = combine_and_save_metrics(metrics_dfs_by_seed, seed_checkpoint_map, dataset_folder_path)

Checking if 'checkpoint' is in 
	checkpoint_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_granite-3.3-8b-instruct.csv
Checking if 'checkpoint' is in 
	checkpoint_mistral-small-3.1.csv
Checking if 'metrics' is in 
	metrics_summary_mistral-small-3.1.csv
Checking if 'checkpoint' is in 
	checkpoint_granite-3.3-8b-instruct.csv
Checking if 'checkpoint' is in 
	checkpoint_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_granite-3.3-8b-instruct.csv
Checking if 'checkpoint' is in 
	checkpoint_mistral-small-3.1.csv
Checking if 'metrics' is in 
	metrics_summary_mistral-small-3.1.csv
Checking if 'checkpoint' is in 
	checkpoint_granite-3.3-8b-instruct.csv
Checking if 'checkpoint' is in 
	checkpoint_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_gemma-3-27b-it.csv
Checking if 'metrics' is in 
	metrics_summary_granite-3.